# Identifying good reviews

Let's create a model to identify helpful reviews that people vote as helpful. They could be low rating reviews,

## Load Data

In [27]:
from pathlib import Path
from evalforge.utils import *

dataset_path = Path("data/clothes_review_10k.jsonl")
data = load_jsonl(dataset_path)

print(f"Number of examples: {len(data)}")
print(f"Number of reviews: {sum(len(example['reviews']) for example in data)}")
print("-"*100)
pprint(data[0])

Number of examples: 9999
Number of reviews: 159447
----------------------------------------------------------------------------------------------------
{
    "parent_asin": "5781728791",
    "main_category": "AMAZON FASHION",
    "title": "Women's Crewneck Striped Shirt Loose Colorblock Sweatshirt Pullover Top",
    "description": [],
    "average_rating": 4.0,
    "rating_number": 27,
    "asin": "5781728791",
    "features": [
        "Denim",
        "Hand Wash Only",
        "Imported",
        "Features: Crewneck, Long Sleeve, Stripe printed,Casual and basic Shirts for Women",
        "Casual pullover tops, perfect to pair with jeans, leggings,denim shorts",
        "This womens everyday loose striped shirt is perfect for Daily Wear, Party, School, Vacation, Office, Work, Home, Club, Night Out. Also a great choice as a gift for your wife, girlfriend, mom, daughter or sisters.",
        "Hand wash or Machine wash:Recommended with cold water"
    ],
    "price": "None",
    "images"

In [28]:
item = data[0]
reviews = item["reviews"]

## LLM

In [67]:
import instructor
from litellm import acompletion

# the import order matters here to have instructor tracing working
import weave

llm_client = instructor.from_litellm(acompletion)

In [50]:
system_prompt = """“”"# Constructing a LLM Judge Benchmark

## The Benchmark
I am trying to build a benchmark for an LLM judge. This benchmark requires positive and negative labels for a given AI-generated output. I have a dataset of Amazon product descriptions and customer reviews which I think I can use. I will construct a benchmark dataset of product descriptions and an associated good / bad ground truth set of labels for the quality of the description text.
I will then pass the product descriptions to a LLM Judge and ask it to rate the descriptions. Then I will compare the Judge’s labels with my ground truth labels in order to understand how aligned.

## Using Reviews As A Proxy signal For Description Text Quality
I want to use the product descriptions as a proxy for AI-generated output and use the feedback provided in the customer review texts as a proxy signal into the quality of the description text.

## Product Description Constraints
However I do not have the actual products in my hand so I am constrained to only assessing the quality of the description text, without knowing if the text matches the actual product in reality. Therefore I am just assessing whether the description text is clear and well-written or whether it is missing information or badly formatted or uses bad english, bad grammar etc. I cannot know if the description text is misleading as I cannot compare to the actual product in reality.

## Scoring Rubric
Given the customer reviews of descriptions please judge the following
- `bad_description`: the product description is missing missing key information or is badly worded, uses poor grammar or is poorly styled or formatted.
- `good_description`: the product description is accurate and helpful.
- `other` the review doesn’t mention the product description or says is misleading.
"""

prompt_template = """The item to review is:

## Item Name
{title}

## Item Description
{description}

## Average Rating
{average_rating}

## Features
{features}

## Review

Rating: {review_rating}
Title: {review_title}

{review_content}
"""


In [51]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewEvaluation(BaseModel):
    annotation: Literal["bad_description", "good_description", "other"] = Field(description="Is the review helpful and provides useful information about the clothing item?")
    note: str = Field(description="Reason for the evaluation")


In [52]:
def format_example(item: dict, review: dict):
    return prompt_template.format(
        title=item["title"],
        description=listify(item["description"]),
        features=listify(item["features"]),
        average_rating=item["average_rating"],
        review_rating=review["rating"],
        review_title=review["title"],
        review_content=review["text"],
    )

In [53]:
print(format_example(item, reviews[0]))

The item to review is:

## Item Name
Cali1850 Women’s Y2K Cargo Pants – Mid Rise Loose Wide Leg Baggy Casual Streetwear Trousers

## Item Description
- None

## Average Rating
4.2

## Features
- 100% Cotton
- Imported
- Zipper closure
- Machine Wash
- HIGH QUALITY – Stay comfortable with style all day long in these casual cargo pants. These trendy bottoms comes in a breathable material that keeps you cool, comfy, and sweat-free in warm weather. The premium quality fabric feels soft and smooth on the skin. Made to let you move freely and comfortably, these long-lasting durable pants will last through multiple washes and wears for many years to come. It’s perfect to wear for all kinds of activities.
- FASHION DESIGN – Have some fun in this wide leg pant. It features a classic zipper fly and button closure, 2 front pockets, 3 side utility pockets, loose relaxed fit with wide leg design to full length. The simple yet stylish design gives you a chic look that allows for range of motion to p

## Helpful reviews

Let's look at the helpful reviews

In [54]:
HELPFUL_VOTE_THRESHOLD = 10

In [55]:
helpful_reviews = []

for item in data:
    for review in item["reviews"]:
        if review["helpful_vote"] > HELPFUL_VOTE_THRESHOLD:
            item_without_reviews = {k: v for k, v in item.items() if k != "reviews"}
            helpful_reviews.append({"item":item_without_reviews, "review": review})

print(f"Number of helpful reviews: {len(helpful_reviews)}")


Number of helpful reviews: 1776


In [68]:
import asyncio
from tqdm.asyncio import tqdm


async def async_map(func, items, max_concurrent=5, desc="Processing"):
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def wrapped_func(**item):
        async with semaphore:
            return await func(**item)
    
    tasks = [wrapped_func(**item) for item in items]
    return await tqdm.gather(*tasks, desc=desc)

@weave.op
async def evaluate_review(item, review):
    """Evaluate a single review using the LLM"""
    review_evaluation = await llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt}, 
            {"role": "user", "content": format_example(item, review)}],
        response_model=ReviewEvaluation,
    )
    return {"item": item, "review": review, "review_evaluation": review_evaluation}

## Dataset

In [69]:
weave.init("amazon_fashion")


# evaluation = weave.Evaluation(dataset = helpful_reviews[:10])
# await evaluation.evaluate(evaluate_review)

annotations = await async_map(evaluate_review, helpful_reviews[:500], max_concurrent=25, desc="Processing reviews")

Processing reviews:   1%|          | 6/500 [00:01<01:23,  5.94it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fae-7d01-b8ba-fd481a6b652c
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fce-7c00-adc0-6a31fe194b3b
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fe3-7753-bfce-0a7f54940427
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd6-7780-9c56-d407596abc75
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fac-77b2-a824-a39fac45322e
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fc8-7600-827c-50291861b7d8
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd0-7c02-87ca-cf957ab6834f
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fe2-7113-9782-573e3d02c99a


Processing reviews:   3%|▎         | 16/500 [00:01<00:26, 17.94it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fca-72d1-b4b9-7b8377e5d3a3
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fe9-7902-b03d-366e217d492a
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fc6-7c03-98df-797ff8239714
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fcd-7740-92ef-05d2108adbc9
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fe5-7863-9435-24da7e612078
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fb0-79d3-893e-a7aecd29d6cd
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd7-7473-b2c2-c93e2973c1ae
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fed-7782-9fad-f527f0ca7bdd
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd5-70b0-a9ae-207d9aebd332
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd2-7f31-94ef-b167c25e122e
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-2fd3-7af0-b00b-50828a1492b4


Processing reviews:   4%|▍         | 21/500 [00:01<00:22, 21.72it/s]

...

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-9258-7532-ab95-39a5cbc922c3
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-928e-7720-a0b9-c738fe15b7ef
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-9274-7d80-8cb1-24fd9527655e
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-8c90-78c0-a021-16fd7429cf77


Processing reviews: 100%|█████████▉| 498/500 [00:26<00:00, 15.62it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-928b-7e01-af5a-ffd635328968
🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-8aa4-7262-aed9-b4d07e5a0515


Processing reviews: 100%|██████████| 500/500 [00:27<00:00, 18.20it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/0192e363-8ec7-7ce0-b355-573322c55475


In [79]:
save_jsonl(annotations, "data/helpful_reviews_annotations.jsonl")